In [1]:
import importlib
import tisd_engine
importlib.reload(tisd_engine) # This forces the notebook to read the file again
from tisd_engine import chat_with_tisd

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# Cell 1
import json
import pandas as pd
from tisd_engine import chat_with_tisd

# Define the test path
TEST_DATA_FILE = "../data/test_set.json"

# Load our test set
with open(TEST_DATA_FILE, "r") as f:
    test_questions = json.load(f)

print(f"Loaded {len(test_questions)} questions for evaluation.")

Loaded 3 questions for evaluation.


In [8]:
# Cell 2
def calculate_similarity(pred, target):
    """Measures how many words overlap between prediction and ground truth."""
    pred_set = set(pred.lower().split())
    target_set = set(target.lower().split())
    
    if len(target_set) == 0: return 0
    
    intersection = pred_set.intersection(target_set)
    return len(intersection) / len(target_set)

# In Notebook 06
from sentence_transformers import SentenceTransformer, util
eval_model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_semantic_score(pred, target):
    emb1 = eval_model.encode(pred, convert_to_tensor=True)
    emb2 = eval_model.encode(target, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()
    
def run_evaluation():
    results = []
    print("Running Evaluation Loop...")
    
    for item in test_questions:
        q = item["question"]
        gt = item["ground_truth"]
        
        # Call the engine
        answer, context, latency = chat_with_tisd(q)
        
        # Calculate overlap
        score = calculate_similarity(answer, gt)
        
        results.append({
            "question": q,
            "answer": answer,
            "score": score
        })
        
        # In your evaluation loop
        answer, context, latency = chat_with_tisd(q)
        print(f"Context: {context[:100]}...") # Print first 100 chars
        
    return pd.DataFrame(results)

# Run it
df_results = run_evaluation()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running Evaluation Loop...


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: day? Discuss these changes with your friends and make a list. Can you guess the brightest object in ...


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: Our Wondrous World | Class 3 60 61 Let us reflect A. Discuss 1. What would happen if there were no p...


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: sense of space, movement and anticipation. Note for the teacher Unit 1.indd 28Unit 1.indd 28 03-Apr-...


In [4]:
# Cell 3
# Calculate the average performance
avg_score = df_results['score'].mean()

print(f"--- EVALUATION SUMMARY ---")
print(f"Average Accuracy Score: {avg_score:.2%}")
print("-" * 30)

# Display the table cleanly
from IPython.display import display
display(df_results)

--- EVALUATION SUMMARY ---
Average Accuracy Score: 42.88%
------------------------------


,question,answer,score
0,What is the sun?,The sun is an object in space that provides li...,0.615385
1,What do plants need to grow?,The information given above provides only the ...,0.250000
2,What are sense organs?,Sure! Here's your answer based on the given in...,0.421053


In [5]:
# Cell 4
# Show questions where the AI scored less than 30%
low_performers = df_results[df_results['score'] < 0.3]

print(f"Questions needing improvement:")
display(low_performers)

Questions needing improvement:


,question,answer,score
1,What do plants need to grow?,The information given above provides only the ...,0.25


In [6]:
from tisd_engine import chat_with_tisd

# Test Question 1
q1 = "What is the sun?"
ans, ctx, lat = chat_with_tisd(q1)

print("=== DIAGNOSTIC TEST 1: 'What is the sun?' ===")
print(f"1. RETRIEVED CONTEXT:\n{ctx}\n")
print(f"2. MODEL ANSWER:\n{ans}\n")
print("="*45)

# Test Question 2
q2 = "What are sense organs?"
ans2, ctx2, lat2 = chat_with_tisd(q2)

print("\n=== DIAGNOSTIC TEST 2: 'What are sense organs?' ===")
print(f"1. RETRIEVED CONTEXT:\n{ctx2}\n")
print(f"2. MODEL ANSWER:\n{ans2}\n")
print("="*45)

Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== DIAGNOSTIC TEST 1: 'What is the sun?' ===
1. RETRIEVED CONTEXT:
day? Discuss these changes with your friends and make a list. Can you guess the brightest object in the sky? It is the Sun! In fact, it is so bright that the other stars cannot be seen when it is present in the sky. The Sun gives us light and heat. Chapter 10.indd 153Chapter 10.indd 153 30-03-2025 09:53:2830-03-2025 09:53:28 Reprint 2026-27 day? Discuss these changes with your friends and make a list. Can you guess the brightest object in the sky? It is the Sun! In fact, it is so bright that the other stars cannot be seen when it is present in the sky. The Sun gives us light and heat. Chapter 10.indd 153Chapter 10.indd 153 30-03-2025 09:53:2830-03-2025 09:53:28 Reprint 2026-27

2. MODEL ANSWER:
The sun is an object in space that provides light and heat to Earth.

Correct  The sun is the brightest object in the sky.

Incorrect answers: The moon or any other celestial body.

Reason for incorrect answers: The question


=

In [9]:
# In Notebook 06
from sentence_transformers import SentenceTransformer, util
eval_model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_semantic_score(pred, target):
    emb1 = eval_model.encode(pred, convert_to_tensor=True)
    emb2 = eval_model.encode(target, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
